# Lesson 25: Popular LLMs (Claude, LLaMA, Gemini…) and how to use LLM APIs 🔌
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Compare popular LLM families: who makes them, **open-weights vs closed**, strengths
2. Run open models locally and compare them fairly
3. Use an LLM **API**: messages and roles, tokens and cost, streaming, JSON output
4. Call the **OpenAI**, **Hugging Face**, **Gemini** and **Claude** APIs safely with keys stored in Colab Secrets

**⏱️ Timing:** the landscape 10' | local comparison 15' | API basics 20' | commercial APIs (teacher demo) 15'

> ⚙️ **Before you start:** go to **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom.

> 🔐 **API keys are passwords.** Never paste a key into a code cell, a screenshot or GitHub. In Colab, click the 🔑 **Secrets** icon in the left sidebar, add the key there, and read it with `userdata.get(...)`.
> Most commercial AI APIs require users to be **18 or older**, so in class the **teacher** owns the key. Every exercise in this notebook also works with the free local model, with no key at all.

## 🗺️ The LLM landscape
| Family | Made by | Weights | Known for |
|---|---|---|---|
| **Claude** | Anthropic | Closed (use via app or API) | Tiers from fast to most capable (Haiku → Sonnet → Opus); long documents, coding, careful safety training ("Constitutional AI") |
| **Gemini** | Google DeepMind | Closed (smaller **Gemma** models are open) | Built to be multimodal (text, images, audio, video); connected to Google products |
| **GPT** | OpenAI | Mostly closed (some open-weight releases) | ChatGPT; very widely used API |
| **LLaMA** | Meta | **Open weights** (with a licence) | You can download and run or fine-tune it yourself |
| **Qwen**, **Mistral**, **DeepSeek**… | Alibaba, Mistral AI, DeepSeek… | Many open weights | Strong small models, multilingual |

> ⚠️ Model names and versions change every few months. Always check the official pages (e.g. https://docs.claude.com, https://ai.google.dev, https://platform.openai.com/docs, https://www.llama.com) for the current line-up.

**Open-weights vs closed:**
- **Open weights**: download, run offline, fine-tune, keep data private; you need your own hardware
- **Closed (API)**: usually the most capable, nothing to install; you pay per token and send your data to the provider

## Part 1: Run two open models locally and compare 🏁

In [ ]:
# 🦙 Start a free LLM server *inside* Colab (Ollama). No API key or account needed.
# ⏳ First run: about 2–3 minutes (installs Ollama and downloads the model).
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

MODEL = "qwen2.5:3b"
!ollama pull {MODEL} > /dev/null 2>&1
!ollama pull llama3.2:1b > /dev/null 2>&1        # Meta's LLaMA, 1 billion parameters
print("✅ Ollama is running with", MODEL)

In [ ]:
!pip -q install ollama
import ollama, time

def chat_local(model, prompt):
    start = time.time()
    r = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
    return r["message"]["content"], time.time() - start

tests = ["What is 17 × 23? Answer with just the number.",
         "Write a haiku about a school bus.",
         "Translate into French: 'Where is the library?'",
         "Who was the first person to walk on Mars?"]          # trick question!
for prompt in tests:
    print(f"❓ {prompt}")
    for m in ["llama3.2:1b", MODEL]:
        answer, secs = chat_local(m, prompt)
        print(f"   [{m}, {secs:.1f}s] {answer.strip()[:200]}")
    print()

### 🧩 Exercise 1: Fair testing
1. Score each answer 0–2. Which model wins? (17 × 23 = 391.)
2. Is one round of 4 questions a **fair** test? What would make it fairer? (More questions, several runs, blind marking…) Real leaderboards try to do this — and still get argued about!

## Part 2: How an LLM API works 📡
Almost every chat API uses the same shape (popularised by OpenAI): a list of **messages**, each with a **role**:
- `system` — background instructions
- `user` — what the person says
- `assistant` — what the model said before (this is how it "remembers" the conversation)

Ollama speaks this **OpenAI-compatible** format, so we can learn with the official `openai` library and no key.

In [ ]:
!pip -q install openai
from openai import OpenAI
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")   # local server; the key is ignored

messages = [
    {"role": "system", "content": "You are a friendly science tutor for 15-year-olds. Keep answers under 60 words."},
    {"role": "user", "content": "Why do we have seasons?"},
]
r = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.3)
print(r.choices[0].message.content)
print("\nTokens used →", "prompt:", r.usage.prompt_tokens, "| answer:", r.usage.completion_tokens)

In [ ]:
# The API has NO memory: to continue the conversation, send the whole history again
messages.append({"role": "assistant", "content": r.choices[0].message.content})
messages.append({"role": "user", "content": "So is it summer in Australia when it's winter in Canada?"})
r2 = client.chat.completions.create(model=MODEL, messages=messages)
print(r2.choices[0].message.content)
print("Prompt tokens this time:", r2.usage.prompt_tokens, "← grows as the chat gets longer")

In [ ]:
# Streaming: print the answer as it's generated (that's the typing effect in chat apps)
stream = client.chat.completions.create(model=MODEL, stream=True,
    messages=[{"role": "user", "content": "Tell me 3 fun facts about octopuses."}])
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="", flush=True)

In [ ]:
# Structured output: ask for JSON so your program can use the answer
import json
r = client.chat.completions.create(
    model=MODEL, temperature=0,
    response_format={"type": "json_object"},
    messages=[{"role": "user", "content": "Give a JSON object with keys 'animal', 'habitat', 'fun_fact' about a kiwi bird."}])
data = json.loads(r.choices[0].message.content)
print(data)
print("Habitat only:", data.get("habitat"))

In [ ]:
# Cost estimate: commercial APIs charge per million tokens (prices vary a lot — check the provider's pricing page)
def cost(prompt_tokens, answer_tokens, price_in_per_million, price_out_per_million):
    return prompt_tokens / 1e6 * price_in_per_million + answer_tokens / 1e6 * price_out_per_million

# Example with made-up prices: $1 per million input tokens, $5 per million output tokens
per_chat = cost(500, 300, 1, 5)
print(f"One chat ≈ ${per_chat:.4f}.  A school of 800 students × 20 chats/day × 200 days ≈ ${per_chat * 800 * 20 * 200:,.0f}/year")

### 🧩 Exercise 2
1. Change the `system` message so the tutor answers only in questions (Socratic style).
2. Ask for JSON describing **3 planets** as a list. Parse it and print a table.
3. Why does a long conversation cost more per message?

## Part 3: Commercial APIs (teacher demo) 🔐
Each cell **skips itself** if the key isn't in Colab Secrets. The code shape is almost the same for every provider — once you know one API, you know them all.

> 💡 Model names below were correct when this notebook was written; check each provider's docs for the current ones.

In [ ]:
from google.colab import userdata

def get_key(name):
    try:
        return userdata.get(name)
    except Exception:
        print(f"⏭️  No {name} in Colab Secrets — skipping this demo.")
        return None

In [ ]:
# OpenAI — same code as the local example, just a different base URL and a real key
key = get_key("OPENAI_API_KEY")
if key:
    oa = OpenAI(api_key=key)
    r = oa.chat.completions.create(model="gpt-4o-mini",      # 👉 check platform.openai.com/docs/models for current names
                                   messages=[{"role": "user", "content": "In one sentence, what is an LLM?"}])
    print(r.choices[0].message.content)

In [ ]:
# Anthropic Claude — docs: https://docs.claude.com/en/api/overview
key = get_key("ANTHROPIC_API_KEY")
if key:
    !pip -q install anthropic
    import anthropic
    claude = anthropic.Anthropic(api_key=key)
    msg = claude.messages.create(
        model="claude-haiku-5-5",        # 👉 the fast tier; see docs.claude.com for the current model list
        max_tokens=200,
        system="You are a friendly tutor for teenagers.",
        messages=[{"role": "user", "content": "In one sentence, what is an LLM?"}])
    print(msg.content[0].text)

In [ ]:
# Google Gemini — docs: https://ai.google.dev
key = get_key("GEMINI_API_KEY")
if key:
    !pip -q install google-genai
    from google import genai
    gem = genai.Client(api_key=key)
    r = gem.models.generate_content(model="gemini-2.5-flash",   # 👉 check ai.google.dev for current model names
                                    contents="In one sentence, what is an LLM?")
    print(r.text)

In [ ]:
# Hugging Face Inference — run models hosted by Hugging Face (free tier with a token from huggingface.co/settings/tokens)
key = get_key("HF_TOKEN")
if key:
    from huggingface_hub import InferenceClient
    hf = InferenceClient(token=key)
    r = hf.chat_completion(model="meta-llama/Llama-3.1-8B-Instruct",
                           messages=[{"role": "user", "content": "In one sentence, what is an LLM?"}], max_tokens=80)
    print(r.choices[0].message.content)

### 🧩 Exercise 3: Spot the pattern
Compare the four commercial API cells. Fill in a table: where does the **model name** go? The **system prompt**? Where is the **answer text** in the response?

### ✅ API safety checklist
- [ ] Keys live in **Secrets / environment variables**, never in code, screenshots or GitHub
- [ ] Set a **spending limit** in the provider dashboard
- [ ] Don't send **personal or private data** you wouldn't want stored by a company
- [ ] Check the provider's **age requirements and terms** — most require users to be 18+

## 📌 Summary
- Major families: **Claude** (Anthropic), **Gemini** (Google), **GPT** (OpenAI), **LLaMA** (Meta, open weights) and many more open models
- APIs use **messages with roles**; they don't remember — you resend the history
- You pay per **token**; use **streaming** for speed and **JSON** for structured answers
- Keep keys **secret**